# Layered Reference Visualization And Checks

This notebook inspects the new layered reference tables and creates simple publication/checking artifacts around nutrient, product/processing, FoodAtlas compound, and graph-feature coverage.

In [1]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT / 'outputs'
REFERENCE = OUT / 'reference'
VIS = OUT / 'visualizations'
VIS.mkdir(parents=True, exist_ok=True)

## Load Reference Tables

In [2]:
feature = pd.read_csv(REFERENCE / 'canonical_food_feature_matrix.csv')
processing = pd.read_csv(REFERENCE / 'canonical_food_processing_reference.csv')
compounds = pd.read_csv(REFERENCE / 'canonical_food_compound_reference.csv')
graph_features = pd.read_csv(REFERENCE / 'canonical_food_graph_features.csv')
summary = json.loads((REFERENCE / 'reference_build_summary.json').read_text())
summary

{'outputs': {'canonical_food_master': '/Users/elchananmycheaux/Library/CloudStorage/OneDrive-Personal/Agens_Intelligens/Agent_Codex/Diet_Data_enhancement/outputs/reference/canonical_food_master.csv',
  'canonical_food_nutrient_reference': '/Users/elchananmycheaux/Library/CloudStorage/OneDrive-Personal/Agens_Intelligens/Agent_Codex/Diet_Data_enhancement/outputs/reference/canonical_food_nutrient_reference.csv',
  'canonical_food_processing_reference': '/Users/elchananmycheaux/Library/CloudStorage/OneDrive-Personal/Agens_Intelligens/Agent_Codex/Diet_Data_enhancement/outputs/reference/canonical_food_processing_reference.csv',
  'canonical_food_compound_reference': '/Users/elchananmycheaux/Library/CloudStorage/OneDrive-Personal/Agens_Intelligens/Agent_Codex/Diet_Data_enhancement/outputs/reference/canonical_food_compound_reference.csv',
  'canonical_food_graph_features': '/Users/elchananmycheaux/Library/CloudStorage/OneDrive-Personal/Agens_Intelligens/Agent_Codex/Diet_Data_enhancement/output

## Layer Coverage Overview

In [3]:
coverage = pd.DataFrame([
    {'layer': 'Canonical foods', 'count': feature['canonical_food_id'].nunique()},
    {'layer': 'OpenFoodFacts product matches', 'count': int(processing['canonical_food_id'].nunique()) if not processing.empty else 0},
    {'layer': 'FoodAtlas compound-linked foods', 'count': int(graph_features.loc[graph_features['foodatlas_compound_count'] > 0, 'canonical_food_id'].nunique()) if not graph_features.empty else 0},
    {'layer': 'FoodAtlas compound links', 'count': int(len(compounds)) if not compounds.empty else 0},
])
fig = px.bar(coverage, x='layer', y='count', text='count', title='Layered Reference Coverage')
fig.update_layout(xaxis_title='', yaxis_title='Count')
fig.write_html(VIS / 'layered_reference_coverage_plotly.html')
coverage

,layer,count
0,Canonical foods,842
1,OpenFoodFacts product matches,842
2,FoodAtlas compound-linked foods,340
3,FoodAtlas compound links,47794


## Top Foods By FoodAtlas Compound Count

In [4]:
top_compound = feature[['canonical_name', 'canonical_category', 'foodatlas_compound_count', 'foodatlas_positive_disease_edge_count', 'foodatlas_negative_disease_edge_count']].sort_values('foodatlas_compound_count', ascending=False).head(30)
fig = px.bar(top_compound, x='canonical_name', y='foodatlas_compound_count', color='canonical_category', title='Top Canonical Foods By FoodAtlas Compound Count')
fig.update_layout(xaxis_title='', yaxis_title='FoodAtlas compounds')
fig.write_html(VIS / 'layered_foodatlas_compound_top_foods.html')
top_compound

,canonical_name,canonical_category,foodatlas_compound_count,foodatlas_positive_disease_edge_count,foodatlas_negative_disease_edge_count
323,Goats Milk,"milk, cream cheese and yogurts",1511,4291,2660
458,Milk,"milk, cream cheese and yogurts",1511,4291,2660
632,Rice crackers,Bread,871,2061,1602
633,Rice crackers,Bread_wholewheat,871,2061,1602
637,Rice paper,"Pasta, Grains and Side dishes",871,2061,1602
636,Rice drink,"milk, cream cheese and yogurts",871,2061,1602
635,Rice crackers,sweets,871,2061,1602
634,Rice crackers,"Pasta, Grains and Side dishes",871,2061,1602
629,Rice,"Pasta, Grains and Side dishes",871,2061,1602
631,Rice Noodles,"Pasta, Grains and Side dishes",871,2061,1602


## Processing Feature Checks

In [5]:
processing_cols = [c for c in ['canonical_name', 'openfoodfacts_product_name', 'brands', 'openfoodfacts_match_score', 'openfoodfacts_confidence', 'ingredient_count', 'additives_n', 'nova_group', 'nutriscore_grade'] if c in processing.columns]
processing[processing_cols].sort_values(['openfoodfacts_confidence', 'openfoodfacts_match_score'], ascending=[True, False]).head(30)

,canonical_name,openfoodfacts_product_name,brands,openfoodfacts_match_score,openfoodfacts_confidence,ingredient_count,additives_n,nova_group,nutriscore_grade
0,Acai,Açai,NaN,1.0,high,1.0,NaN,NaN,unknown
3,Alfalfa sprouts,Alfalfa Sprouts,NaN,1.0,high,1.0,NaN,NaN,unknown
4,Almond Beverage,Almond beverage,Silk,1.0,high,11.0,3.0,4.0,unknown
5,Almond flour,ALMOND FLOUR,simple truth,1.0,high,1.0,NaN,NaN,unknown
7,Almonds,Almonds,NaN,1.0,high,6.0,0.0,3.0,d
10,Aperol,Aperol,NaN,1.0,high,1.0,NaN,NaN,unknown
11,Apple,Apple,Manzanita Sol,1.0,high,7.0,4.0,4.0,e
13,Apple juice,Apple Juice,Florida's Natural,1.0,high,3.0,0.0,1.0,a
14,Apple Vinegar,Apple Vinegar,American Garden,1.0,high,1.0,NaN,NaN,unknown
15,Applesauce,Applesauce,The Kroger Co.,1.0,high,5.0,0.0,4.0,b


## Save Compact Visualization Summary

In [6]:
viz_summary = {
    'feature_matrix_rows': int(feature.shape[0]),
    'feature_matrix_columns': int(feature.shape[1]),
    'openfoodfacts_processing_rows': int(processing.shape[0]),
    'foodatlas_compound_reference_rows': int(compounds.shape[0]),
    'outputs': {
        'layered_reference_coverage_plotly': str(VIS / 'layered_reference_coverage_plotly.html'),
        'layered_foodatlas_compound_top_foods': str(VIS / 'layered_foodatlas_compound_top_foods.html'),
    }
}
(VIS / 'layered_reference_visualization_summary.json').write_text(json.dumps(viz_summary, indent=2))
viz_summary

{'feature_matrix_rows': 842,
 'feature_matrix_columns': 228,
 'openfoodfacts_processing_rows': 842,
 'foodatlas_compound_reference_rows': 47794,
 'outputs': {'layered_reference_coverage_plotly': '/Users/elchananmycheaux/Library/CloudStorage/OneDrive-Personal/Agens_Intelligens/Agent_Codex/Diet_Data_enhancement/outputs/visualizations/layered_reference_coverage_plotly.html',
  'layered_foodatlas_compound_top_foods': '/Users/elchananmycheaux/Library/CloudStorage/OneDrive-Personal/Agens_Intelligens/Agent_Codex/Diet_Data_enhancement/outputs/visualizations/layered_foodatlas_compound_top_foods.html'}}